# Archive Selection - EDA Analysis Report

A comprehensive data-driven analysis of archive fashion items tracked on Grailed, covering supply/demand dynamics, turnover velocity, hype signals, and pricing patterns.

**Data Sources:**
- `scorecard.csv` - Composite scoring with 4-dimension metrics
- `grailed_sold.csv` - Recent sold transaction records
- `grailed_listings.csv` - Current active listings with follower counts

---

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings("ignore")

# Load data
scorecard = pd.read_csv("scorecard.csv")
sold = pd.read_csv("grailed_sold.csv")
listings = pd.read_csv("grailed_listings.csv")

# Parse dates
scorecard["calc_date"] = pd.to_datetime(scorecard["calc_date"])
sold["sold_at"] = pd.to_datetime(sold["sold_at"], errors="coerce")
sold["fetch_date"] = pd.to_datetime(sold["fetch_date"], errors="coerce")
listings["created_at"] = pd.to_datetime(listings["created_at"], errors="coerce")
listings["fetch_date"] = pd.to_datetime(listings["fetch_date"], errors="coerce")

print("Data loaded successfully.")

## 1. Data Overview

A quick summary of the dataset scope: how many items we're tracking, data volume, and time coverage.

In [ ]:
n_keywords = scorecard["keyword"].nunique()
n_listings = len(listings)
n_sold = len(sold)
n_brands = scorecard["keyword"].apply(lambda x: " ".join(x.split()[:2]) if len(x.split()) >= 2 else x).nunique()

# Time ranges
listing_range = f"{listings['created_at'].min().strftime('%Y-%m-%d')} to {listings['fetch_date'].max().strftime('%Y-%m-%d')}"
sold_range = f"{sold['sold_at'].min().strftime('%Y-%m-%d')} to {sold['sold_at'].max().strftime('%Y-%m-%d')}"
scorecard_date = scorecard["calc_date"].max().strftime("%Y-%m-%d")

overview = pd.DataFrame({
    "Metric": [
        "Tracked Items (Keywords)",
        "Brands Covered",
        "Active Listings",
        "Sold Records (30d)",
        "Listings Date Range",
        "Sold Date Range",
        "Scorecard Date",
    ],
    "Value": [
        n_keywords,
        n_brands,
        f"{n_listings:,}",
        n_sold,
        listing_range,
        sold_range,
        scorecard_date,
    ]
})

print("=" * 50)
print("       DATA OVERVIEW")
print("=" * 50)
for _, row in overview.iterrows():
    print(f"  {row['Metric']:<28} {row['Value']}")
print("=" * 50)

## 2. Supply / Demand Analysis

Supply/Demand ratio = active listings / 30-day sold count. A lower ratio means higher scarcity.

- **Scarce (S/D < 30):** Demand outpaces supply — strong resale potential
- **Balanced (30-100):** Healthy market equilibrium  
- **Oversupply (S/D > 100):** Too many listings relative to demand — harder to sell

In [ ]:
sd = scorecard.dropna(subset=["supply_demand_ratio"]).sort_values("supply_demand_ratio").copy()
sd["short_name"] = sd["keyword"].apply(lambda x: x if len(x) <= 28 else x[:26] + "...")

# Color by zone
def sd_zone(r):
    if r < 30:
        return "Scarce (< 30)"
    elif r <= 100:
        return "Balanced (30-100)"
    else:
        return "Oversupply (> 100)"

sd["zone"] = sd["supply_demand_ratio"].apply(sd_zone)
zone_colors = {"Scarce (< 30)": "#059669", "Balanced (30-100)": "#2563eb", "Oversupply (> 100)": "#dc2626"}

fig = px.bar(
    sd, x="supply_demand_ratio", y="short_name", color="zone",
    orientation="h",
    color_discrete_map=zone_colors,
    labels={"supply_demand_ratio": "Supply / Demand Ratio", "short_name": "", "zone": "Zone"},
    title="Supply/Demand Ratio by Item",
)

# Add threshold lines
fig.add_vline(x=30, line_dash="dash", line_color="#059669", annotation_text="Scarce < 30", annotation_position="top right")
fig.add_vline(x=100, line_dash="dash", line_color="#dc2626", annotation_text="Oversupply > 100", annotation_position="top right")

fig.update_layout(
    height=500, yaxis=dict(categoryorder="total ascending"),
    margin=dict(l=0, r=20, t=40, b=20),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
fig.show()

# Summary
scarce_items = sd[sd["supply_demand_ratio"] < 30]["keyword"].tolist()
oversupply_items = sd[sd["supply_demand_ratio"] > 100]["keyword"].tolist()
print(f"\n Scarce items (S/D < 30): {', '.join(scarce_items) if scarce_items else 'None'}")
print(f" Oversupply items (S/D > 100): {', '.join(oversupply_items) if oversupply_items else 'None'}")

## 3. Turnover Velocity Analysis

Which items are selling the fastest? 30-day sold count is the most direct indicator of market liquidity — higher velocity means easier to flip.

In [ ]:
vel = scorecard.dropna(subset=["velocity_30d"]).sort_values("velocity_30d", ascending=True).copy()
vel["short_name"] = vel["keyword"].apply(lambda x: x if len(x) <= 28 else x[:26] + "...")

fig = go.Figure(go.Bar(
    x=vel["velocity_30d"],
    y=vel["short_name"],
    orientation="h",
    text=vel["velocity_30d"].astype(int),
    textposition="outside",
    marker=dict(
        color=vel["velocity_30d"],
        colorscale=[[0, "#bfdbfe"], [1, "#1d4ed8"]],
    ),
))

fig.update_layout(
    title="30-Day Sold Count by Item (Velocity)",
    xaxis_title="Sold in Last 30 Days",
    height=500,
    margin=dict(l=0, r=40, t=40, b=20),
)
fig.show()

# Analysis
top_vel = vel.iloc[-1]
zero_vel = vel[vel["velocity_30d"] == 0]["keyword"].tolist()
print(f"\n Fastest turnover: {top_vel['keyword']} ({int(top_vel['velocity_30d'])} sales in 30 days)")
if zero_vel:
    print(f" Zero sales (30d): {', '.join(zero_vel)}")
    print("   -> These items may be too niche or overpriced for current market conditions.")

## 4. Hype / Follower Analysis

Average followers per listing measures buyer interest. High followers + low sales = **"watching but not buying"** — these items have latent demand that could convert with the right pricing or market shift.

In [ ]:
hype = scorecard.dropna(subset=["avg_followers", "velocity_30d"]).copy()
hype["short_name"] = hype["keyword"].apply(lambda x: x if len(x) <= 28 else x[:26] + "...")

# Scatter: followers vs velocity, size = total_score
fig = px.scatter(
    hype, x="velocity_30d", y="avg_followers",
    size="total_score", hover_name="keyword",
    text="short_name",
    labels={"velocity_30d": "30-Day Sales", "avg_followers": "Avg Followers per Listing"},
    title="Hype vs Velocity — Identifying 'Watching but Not Buying' Items",
    size_max=40,
)

# Quadrant lines
median_vel = hype["velocity_30d"].median()
median_fol = hype["avg_followers"].median()
fig.add_hline(y=median_fol, line_dash="dot", line_color="#9ca3af", annotation_text="Median Followers")
fig.add_vline(x=median_vel, line_dash="dot", line_color="#9ca3af", annotation_text="Median Sales")

# Highlight quadrant
fig.add_annotation(
    x=0.05, y=0.95, xref="paper", yref="paper",
    text="HIGH HYPE<br>LOW SALES<br>(Potential)", showarrow=False,
    font=dict(size=11, color="#d97706"),
    bgcolor="rgba(217,119,6,0.08)", bordercolor="#d97706", borderwidth=1, borderpad=6,
)

fig.update_traces(textposition="top center", textfont_size=9)
fig.update_layout(height=500, margin=dict(l=0, r=20, t=40, b=20))
fig.show()

# Find high-hype low-velocity items
potential = hype[(hype["avg_followers"] > median_fol) & (hype["velocity_30d"] <= median_vel)]
if not potential.empty:
    print("\n 'High Hype, Low Sales' potential items:")
    for _, r in potential.iterrows():
        print(f"   - {r['keyword']}  (Avg Followers: {r['avg_followers']:.1f}, 30d Sales: {int(r['velocity_30d'])})")

## 5. Price Analysis

Box plots of sold prices by item reveal price range, median, and outliers. This helps set buy/sell thresholds — a tight box means predictable margins, a wide box means higher risk.

In [ ]:
# Use sold data for price distribution
price_df = sold.dropna(subset=["sold_price"]).copy()
price_df["short_name"] = price_df["keyword"].apply(lambda x: x if len(x) <= 28 else x[:26] + "...")

# Order by median sold price
median_order = price_df.groupby("short_name")["sold_price"].median().sort_values().index.tolist()

fig = px.box(
    price_df, x="sold_price", y="short_name",
    orientation="h",
    labels={"sold_price": "Sold Price (USD)", "short_name": ""},
    title="Sold Price Distribution by Item (Box Plot)",
    category_orders={"short_name": median_order},
    color_discrete_sequence=["#2563eb"],
)

fig.update_layout(
    height=500,
    margin=dict(l=0, r=20, t=40, b=20),
)
fig.show()

# Price summary table
price_summary = price_df.groupby("keyword")["sold_price"].agg(["count", "median", "min", "max", "std"]).round(0)
price_summary.columns = ["Sold Count", "Median $", "Min $", "Max $", "Std Dev $"]
price_summary = price_summary.sort_values("Median $", ascending=False)
print("\n Price Summary:")
print(price_summary.to_string())

# Flag high-volatility items
if "Std Dev $" in price_summary.columns:
    high_vol = price_summary[price_summary["Std Dev $"] > price_summary["Median $"] * 0.5]
    if not high_vol.empty:
        print(f"\n High price volatility (Std > 50% of median):")
        for name, row in high_vol.iterrows():
            print(f"   - {name}: Median ${row['Median $']:.0f}, Std ${row['Std Dev $']:.0f}")

## 6. Composite Score Overview

A side-by-side view of all 4 scoring dimensions for every tracked item, stacked to show how each dimension contributes to the final score.

In [ ]:
sc = scorecard.dropna(subset=["total_score"]).sort_values("total_score").copy()
sc["short_name"] = sc["keyword"].apply(lambda x: x if len(x) <= 28 else x[:26] + "...")

# Weighted contributions
sc["w_sd"]  = sc["supply_demand_score"] * 0.35
sc["w_vel"] = sc["velocity_score"] * 0.30
sc["w_hyp"] = sc["grailed_hype_score"] * 0.25
sc["w_mom"] = sc["price_momentum_score"] * 0.10

fig = go.Figure()
dims = [
    ("w_sd",  "Supply/Demand (35%)", "#2563eb"),
    ("w_vel", "Velocity (30%)",      "#059669"),
    ("w_hyp", "Hype (25%)",          "#d97706"),
    ("w_mom", "Momentum (10%)",      "#7c3aed"),
]
for col, name, color in dims:
    fig.add_trace(go.Bar(
        x=sc[col], y=sc["short_name"], orientation="h",
        name=name, marker_color=color,
    ))

fig.update_layout(
    barmode="stack",
    title="Weighted Score Breakdown by Item",
    xaxis_title="Weighted Score Contribution",
    height=500,
    margin=dict(l=0, r=20, t=40, b=20),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
fig.show()

---

## 7. Key Findings & Recommendations

In [ ]:
# Auto-generate key findings from data
sc_sorted = scorecard.dropna(subset=["total_score"]).sort_values("total_score", ascending=False)
top_item = sc_sorted.iloc[0]

best_velocity = scorecard.loc[scorecard["velocity_30d"].idxmax()]
most_scarce = scorecard.dropna(subset=["supply_demand_ratio"])
most_scarce = most_scarce.loc[most_scarce["supply_demand_ratio"].idxmin()]
most_hyped = scorecard.loc[scorecard["avg_followers"].idxmax()]

# High hype low velocity
hype_check = scorecard.dropna(subset=["avg_followers", "velocity_30d"])
med_f = hype_check["avg_followers"].median()
med_v = hype_check["velocity_30d"].median()
sleepers = hype_check[(hype_check["avg_followers"] > med_f) & (hype_check["velocity_30d"] <= med_v)]

from IPython.display import Markdown, display

findings = f"""
### Key Findings

1. **{top_item['keyword']}** is the top-ranked item overall (score: {top_item['total_score']:.2f}/10), driven by strong supply/demand scarcity and high turnover velocity.

2. **{best_velocity['keyword']}** leads in market liquidity with **{int(best_velocity['velocity_30d'])} sales in the last 30 days** — this is the easiest item to flip quickly on Grailed.

3. **{most_scarce['keyword']}** has the lowest supply/demand ratio ({most_scarce['supply_demand_ratio']:.1f}x) — extreme scarcity suggests strong pricing power for sellers.

4. **{most_hyped['keyword']}** has the highest average follower count ({most_hyped['avg_followers']:.1f} per listing) — the strongest buyer intent signal in the dataset.

5. **"Watching but not buying" items** ({len(sleepers)} found): {', '.join(sleepers['keyword'].tolist())} — these have above-median hype but below-median sales. They represent **latent demand** that could convert with strategic pricing or market momentum shifts.

### Recommendations

| Action | Target Items | Rationale |
|--------|-------------|-----------|
| **Buy & flip immediately** | {best_velocity['keyword']} | Highest velocity = fastest ROI cycle |
| **Hold for appreciation** | {most_scarce['keyword']} | Extreme scarcity drives long-term value |
| **Monitor closely** | {', '.join(sleepers['keyword'].head(2).tolist())} | High hype signals potential breakout |
| **Avoid oversupply** | {', '.join(scorecard[scorecard['supply_demand_ratio'] > 100].dropna(subset=['supply_demand_ratio'])['keyword'].tolist()) or 'N/A'} | Too many listings = downward price pressure |
"""

display(Markdown(findings))